# **Chú thích**
**Mục đích cuả file này là dùng để hợp nhất dữ liệu**

Mã nguồn này chứa dữ liệu về Số lượng bài nhạc(audio features) - Số người dùng - Số lượt nghe - lời bài hát
- data: dữ liệu từ spotify có sẵn audio features
- tp: dữ liệu người dùng
- tracks: meta chứa tên bài hát, nghệ sĩ
- lyrics: dữ liệu chứa lời bài hát
- **Data thực hiện như sau:**
  + cắt giảm data xuống còn 20.000 users / 1 triệu users
  + lọc có điều kiện để:
    + user nghe ít nhất 20 bài và nhiều nhất 800 bài
    + bài hát được nghe ít nhất 20 users và nhiều nhất 3000 users
  + merge với tracks với data để lọc ra những bài có audio features
  + TP chứa hành vi người dùng sẽ chỉ lấy những dòng thuộc track_id trong phần merge --> full_data_nolyrics
  + full_data_nolyrics ghép với data thông qua track_id để từng dòng có đủ audio features --> data_audio
  + data_audio ghép với lyrics thông qua track_id để tạo data hoàn chỉnh --> data_lyrics

- **Mở rộng thêm dữ liệu tại**
    + *http://millionsongdataset.com/*
    + *https://www.kaggle.com/datasets/ryanholbrook/the-million-songs-dataset/data*

# **1. Import thư viện cần thiết**

In [1]:
import os
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

import re
import unicodedata

In [2]:
# --- Xuất dữ liệu ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Hàm làm sạch text**

In [3]:
def clean_text(s):
    if pd.isna(s):
        return ''
    s = s.lower()
    # Bỏ dấu tiếng (é -> e, ñ -> n)
    s = ''.join(c for c in unicodedata.normalize('NFD', s)
                if unicodedata.category(c) != 'Mn')
    s = re.sub(r'\(.*?\)', '', s)          # bỏ phần trong ngoặc
    s = re.sub(r'feat\.?|ft\.?|featuring', '', s)  # bỏ feat
    s = re.sub(r'[^a-z0-9\s]', '', s)      # bỏ ký tự đặc biệt
    s = re.sub(r'\s+', ' ', s).strip()     # bỏ khoảng trắng thừa
    return s

# **2. Tải dataset Spotify từ Kaggle**

In [4]:
import kagglehub
vatsalmavani_spotify_dataset_path = kagglehub.dataset_download('vatsalmavani/spotify-dataset')
print('Kaggle dataset downloaded:', vatsalmavani_spotify_dataset_path)

100%|██████████| 16.5M/16.5M [00:01<00:00, 12.6MB/s]

Extracting files...


Kaggle dataset downloaded: /root/.cache/kagglehub/datasets/vatsalmavani/spotify-dataset/versions/1


**Đọc 3 file CSV chính**

In [5]:
data = pd.read_csv(f"{vatsalmavani_spotify_dataset_path}/data/data.csv")
genre_data = pd.read_csv(f"{vatsalmavani_spotify_dataset_path}/data/data_by_genres.csv")
year_data = pd.read_csv(f"{vatsalmavani_spotify_dataset_path}/data/data_by_year.csv")

print("Data shapes:", data.shape, genre_data.shape, year_data.shape)

Data shapes: (170653, 19) (2973, 14) (100, 14)


In [6]:
data = data.drop_duplicates(subset=["id"])

In [7]:
data.shape

(170653, 19)

## **2.1. Thêm cột duration (s)**

In [8]:
data['duration_s'] = data['duration_ms'] / 1000.0

## **2.2. Parse artists từ chuỗi sang list và chuẩn hóa**

In [9]:
# Tách danh sách nghệ sĩ
data['artists_list'] = data['artists'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

# Lấy nghệ sĩ chính
data['main_artist'] = data['artists_list'].apply(lambda x: x[0] if len(x) > 0 else None)

# Chuẩn hóa
data['name_clean'] = data['name'].apply(clean_text)
data['artist_clean'] = data['main_artist'].apply(clean_text)

## **2.3. Thêm cột decade (thập niên)**

In [10]:
def get_decade(year):
    return f"{int(year//10)*10}s"
data['decade'] = data['year'].apply(get_decade)

## **2.4. Kiểm tra lỗi tempo = 0 và bài quá dài/ngắn**

In [11]:
tempo_zero_count = (data['tempo'] == 0).sum()
very_long_count = (data['duration_s'] > 3600).sum()  # > 1 giờ
very_short_count = (data['duration_s'] < 60).sum()  # < 1 phút
print(f"tempo == 0: {tempo_zero_count} bài, >1h: {very_long_count} bài, <1p: {very_short_count}")

tempo == 0: 143 bài, >1h: 7 bài, <1p: 1615


**Loại bỏ lỗi tempo = 0 và bài hát có 1p>x>1h**

In [12]:
data = data[
    (data['tempo'] != 0) &
    (data['duration_ms'] >= 60_000) &
    (data['duration_ms'] <= 60 * 60 * 1000)
]

print(f"Sau khi làm sạch còn {len(data)} bài hát hợp lệ.")

Sau khi làm sạch còn 168915 bài hát hợp lệ.


## **2.5. Kiểm tra và loại bỏ cột không dùng**

In [13]:
data.columns

Index(['valence', 'year', 'acousticness', 'artists', 'danceability',
       'duration_ms', 'energy', 'explicit', 'id', 'instrumentalness', 'key',
       'liveness', 'loudness', 'mode', 'name', 'popularity', 'release_date',
       'speechiness', 'tempo', 'duration_s', 'artists_list', 'main_artist',
       'name_clean', 'artist_clean', 'decade'],
      dtype='object')

In [14]:
cols_to_drop = ["id", "release_date", "artists", "name", "duration_ms", "key", "mode", "artists_list"]  # ví dụ

data = data.drop(columns=cols_to_drop)

In [15]:
data.shape

(168915, 17)

In [16]:
data.head()

,valence,year,acousticness,danceability,energy,explicit,instrumentalness,liveness,loudness,popularity,speechiness,tempo,duration_s,main_artist,name_clean,artist_clean,decade
0,0.0594,1921,0.982,0.279,0.211,0,0.878000,0.665,-20.096,4,0.0366,80.954,831.667,Sergei Rachmaninoff,piano concerto no 3 in d minor op 30 iii final...,sergei rachmaninoff,1920s
1,0.9630,1921,0.732,0.819,0.341,0,0.000000,0.160,-12.441,5,0.4150,60.936,180.533,Dennis Day,clancy lowered the boom,dennis day,1920s
2,0.0394,1921,0.961,0.328,0.166,0,0.913000,0.101,-14.850,5,0.0339,110.339,500.062,KHP Kridhamardawa Karaton Ngayogyakarta Hadini...,gati bali,khp kridhamardawa karaton ngayogyakarta hadini...,1920s
3,0.1650,1921,0.967,0.275,0.309,0,0.000028,0.381,-9.316,3,0.0354,100.109,210.000,Frank Parker,danny boy,frank parker,1920s
4,0.2530,1921,0.957,0.418,0.193,0,0.000002,0.229,-10.096,2,0.0380,101.665,166.693,Phil Regan,when irish eyes are smiling,phil regan,1920s


# **3. Tải Taste Profile Subset (user-song)**

In [17]:
!wget http://millionsongdataset.com/sites/default/files/challenge/train_triplets.txt.zip -P /content/MSD/

import zipfile
zip_path = "/content/MSD/train_triplets.txt.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/MSD/")

--2026-01-07 08:51:18--  http://millionsongdataset.com/sites/default/files/challenge/train_triplets.txt.zip
Resolving millionsongdataset.com (millionsongdataset.com)... 172.104.14.177
Connecting to millionsongdataset.com (millionsongdataset.com)|172.104.14.177|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 512137572 (488M) [application/zip]
Saving to: ‘/content/MSD/train_triplets.txt.zip’

train_triplets.txt. 100%[===================>] 488.41M  11.8MB/s    in 44s     

2026-01-07 08:52:02 (11.2 MB/s) - ‘/content/MSD/train_triplets.txt.zip’ saved [512137572/512137572]



In [18]:
file_path = "/content/MSD/train_triplets.txt"
tp = pd.read_csv(
    file_path,
    sep='\t',
    names=['user_id', 'song_id', 'play_count'])

print("Taste Profile loaded:", tp.shape)

Taste Profile loaded: (48373586, 3)


In [19]:
tp = tp.drop_duplicates(subset=["user_id", "song_id"])

In [20]:
tp.shape

(48373586, 3)

## **3.1. Giảm user**

In [21]:
top_users = (
    tp.groupby('user_id')['song_id']
      .nunique()
      .sort_values(ascending=False)
      .head(20_000)
      .index
)

tp = tp[tp['user_id'].isin(top_users)]

## **3.2. Thêm cột rating**

In [22]:
tp['rating'] = np.log1p(tp['play_count'])

**Kiểm tra dữ liệu**

In [23]:
tp.shape

(6516047, 4)

In [24]:
tp.head()

,user_id,song_id,play_count,rating
558,5a905f000fc1ff3df7ca807d57edb608863db05d,SOADGFH12A8C143D89,11,2.484907
559,5a905f000fc1ff3df7ca807d57edb608863db05d,SOAFOBL12AF72A25BA,12,2.564949
560,5a905f000fc1ff3df7ca807d57edb608863db05d,SOAFTRR12AF72A8D4D,1,0.693147
561,5a905f000fc1ff3df7ca807d57edb608863db05d,SOAIILB12A58A776F7,3,1.386294
562,5a905f000fc1ff3df7ca807d57edb608863db05d,SOAIIYK12A6D4F6666,1,0.693147


## **3.3. Tải mapping**

In [25]:
!wget http://millionsongdataset.com/sites/default/files/AdditionalFiles/unique_tracks.txt -P /content/MSD/

--2026-01-07 08:54:25--  http://millionsongdataset.com/sites/default/files/AdditionalFiles/unique_tracks.txt
Resolving millionsongdataset.com (millionsongdataset.com)... 172.104.14.177
Connecting to millionsongdataset.com (millionsongdataset.com)|172.104.14.177|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84046293 (80M) [text/plain]
Saving to: ‘/content/MSD/unique_tracks.txt’

unique_tracks.txt   100%[===================>]  80.15M  12.8MB/s    in 7.4s    

2026-01-07 08:54:33 (10.8 MB/s) - ‘/content/MSD/unique_tracks.txt’ saved [84046293/84046293]



In [26]:
tracks = pd.read_csv(
    "/content/MSD/unique_tracks.txt",
    sep='<SEP>',
    header=None,
    names=['track_id', 'song_id', 'artist_name', 'title'],
    engine='python'
)

print("Unique tracks loaded:", tracks.shape) #(1000000, 4)

Unique tracks loaded: (1000000, 4)


In [27]:
tracks = tracks.drop_duplicates(subset='song_id', keep='first')
tracks = tracks.dropna(subset=['title'])

In [28]:
tracks.shape

(999039, 4)

## **3.4. Làm sạch mapping**

In [29]:
tracks['title_clean'] = tracks['title'].apply(clean_text)
tracks['artist_clean'] = tracks['artist_name'].apply(clean_text)

In [30]:
tracks.head()

,track_id,song_id,artist_name,title,title_clean,artist_clean
0,TRMMMYQ128F932D901,SOQMMHC12AB0180CB8,Faster Pussy cat,Silent Night,silent night,faster pussy cat
1,TRMMMKD128F425225D,SOVFVAK12A8C1350D9,Karkkiautomaatti,Tanssi vaan,tanssi vaan,karkkiautomaatti
2,TRMMMRX128F93187D9,SOGTUKN12AB017F4F1,Hudson Mohawke,No One Could Ever,no one could ever,hudson mohawke
3,TRMMMCH128F425532C,SOBNYVR12A8C13558C,Yerba Brava,Si Vos Querés,si vos queres,yerba brava
4,TRMMMWA128F426B589,SOHSBXH12A8C13B0DF,Der Mystic,Tangle Of Aspens,tangle of aspens,der mystic


# **4. Tải MXM (lyrics)**

In [31]:
!wget http://millionsongdataset.com/sites/default/files/AdditionalFiles/mxm_dataset_train.txt.zip -P /content/MSD/
!wget http://millionsongdataset.com/sites/default/files/AdditionalFiles/mxm_dataset_test.txt.zip -P /content/MSD/

--2026-01-07 08:54:58--  http://millionsongdataset.com/sites/default/files/AdditionalFiles/mxm_dataset_train.txt.zip
Resolving millionsongdataset.com (millionsongdataset.com)... 172.104.14.177
Connecting to millionsongdataset.com (millionsongdataset.com)|172.104.14.177|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 36502856 (35M) [application/zip]
Saving to: ‘/content/MSD/mxm_dataset_train.txt.zip’

mxm_dataset_train.t 100%[===================>]  34.81M  10.1MB/s    in 3.5s    

2026-01-07 08:55:02 (10.1 MB/s) - ‘/content/MSD/mxm_dataset_train.txt.zip’ saved [36502856/36502856]

--2026-01-07 08:55:02--  http://millionsongdataset.com/sites/default/files/AdditionalFiles/mxm_dataset_test.txt.zip
Resolving millionsongdataset.com (millionsongdataset.com)... 172.104.14.177
Connecting to millionsongdataset.com (millionsongdataset.com)|172.104.14.177|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4798712 (4.6M) [application/zip]
Saving to: ‘/

In [32]:
zip_path = "/content/MSD/mxm_dataset_train.txt.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/MSD/")

zip_path = "/content/MSD/mxm_dataset_test.txt.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/MSD/")

In [33]:
input_files = [
    "/content/MSD/mxm_dataset_train.txt",
    "/content/MSD/mxm_dataset_test.txt"
]

output_path = "/content/MSD/lyrics_track_ids.txt"
chunk_size = 1000

vocab = []

In [34]:
def load_checkpoint(file_path):
    cp = file_path.replace('.txt', '_checkpoint.txt')
    try:
        with open(cp, 'r') as f:
            return int(f.read().strip())
    except:
        return 0

def save_checkpoint(file_path, line_num):
    cp = file_path.replace('.txt', '_checkpoint.txt')
    with open(cp, 'w') as f:
        f.write(str(line_num))

In [35]:
def process_mxm_file(file_path):
    global vocab

    start_line = load_checkpoint(file_path)
    print(f"Resume from line {start_line}")

    buffer = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            if line_num < start_line:
                continue

            line = line.strip()

            if line.startswith('#'):
                continue

            if line.startswith('%'):
                if not vocab:
                    vocab = line[1:].split(',')
                continue

            parts = line.split(',')
            track_id = parts[0]

            # Build pseudo-lyrics
            words = []
            for wc in parts[2:]:
                if ':' not in wc:
                    continue
                wid, cnt = wc.split(':')
                wid = int(wid) - 1
                cnt = int(cnt)

                words.extend([vocab[wid]] * cnt)

            lyrics_text = " ".join(words)

            buffer.append({
                'track_id': track_id,
                'lyrics': lyrics_text
            })

            if len(buffer) >= chunk_size:
                pd.DataFrame(buffer).to_csv(
                    output_path,
                    mode='a',
                    header=not os.path.exists(output_path),
                    index=False
                )
                buffer = []
                save_checkpoint(file_path, line_num)
                print(f"Saved up to line {line_num}")

    if buffer:
        pd.DataFrame(buffer).to_csv(
            output_path,
            mode='a',
            header=not os.path.exists(output_path),
            index=False
        )
        save_checkpoint(file_path, line_num)

In [36]:
for file_path in input_files:
    process_mxm_file(file_path)

lyrics = pd.read_csv(output_path)
print("Total tracks:", len(lyrics))

Resume from line 0
Saved up to line 1017
Saved up to line 2017
Saved up to line 3017
Saved up to line 4017
Saved up to line 5017
Saved up to line 6017
Saved up to line 7017
Saved up to line 8017
Saved up to line 9017
Saved up to line 10017
Saved up to line 11017
Saved up to line 12017
Saved up to line 13017
Saved up to line 14017
Saved up to line 15017
Saved up to line 16017
Saved up to line 17017
Saved up to line 18017
Saved up to line 19017
Saved up to line 20017
Saved up to line 21017
Saved up to line 22017
Saved up to line 23017
Saved up to line 24017
Saved up to line 25017
Saved up to line 26017
Saved up to line 27017
Saved up to line 28017
Saved up to line 29017
Saved up to line 30017
Saved up to line 31017
Saved up to line 32017
Saved up to line 33017
Saved up to line 34017
Saved up to line 35017
Saved up to line 36017
Saved up to line 37017
Saved up to line 38017
Saved up to line 39017
Saved up to line 40017
Saved up to line 41017
Saved up to line 42017
Saved up to line 43017
S

In [37]:
lyrics.shape

(237662, 2)

In [38]:
lyrics.head()

,track_id,lyrics
0,TRAAAAV128F421A322,i i i i i i the the the the you you to to and ...
1,TRAAABD128F429CF47,i i i i i i i i i i you you you you you you yo...
2,TRAAAED128E0783FAB,i i i i i i i i i i i i i i i i i i i i i i i ...
3,TRAAAEF128F4273421,i i i i i the the the the you you you to to an...
4,TRAAAEW128F42930C0,i i i i to to to to to and and and and and and...


# **5. Đồng bộ dữ liệu**

## **5.1. Merge TP với Spotify**

### **5.1.1. Xử lý các bài hát không nằm trong TP data**

In [39]:
# Chuẩn hoá key
data["key"] = data["artist_clean"] + "||" + data["name_clean"]

valid_keys = set(data["key"])
print("Valid songs in data:", len(valid_keys))

Valid songs in data: 150540


In [40]:
tracks["key"] = tracks["artist_clean"] + "||" + tracks["title_clean"]

# Lọc tracks chỉ còn bài có trong data
tracks = tracks[tracks["key"].isin(valid_keys)]

print("Tracks after filter:", len(tracks))

Tracks after filter: 51118


In [41]:
song_map = tracks.set_index("song_id")[
    ["track_id", "artist_name", "title", "artist_clean", "title_clean"]
].to_dict("index")

print("Song map size:", len(song_map))

Song map size: 51118


In [42]:
chunksize = 1_000_000
output_file = "/content/drive/MyDrive/datasets/full_data_nolyrics.csv"

first_chunk = True
total = len(tp)

for i, start in enumerate(range(0, total, chunksize), start=1):
    print(f"Processing chunk {i}")

    chunk = tp.iloc[start:start+chunksize]

    filtered = chunk[chunk["song_id"].isin(song_map)]

    if filtered.empty:
        continue

    meta = filtered["song_id"].map(song_map)

    filtered = filtered.assign(
        track_id = meta.apply(lambda x: x["track_id"]),
        artist_name = meta.apply(lambda x: x["artist_name"]),
        title = meta.apply(lambda x: x["title"]),
        artist_clean = meta.apply(lambda x: x["artist_clean"]),
        title_clean = meta.apply(lambda x: x["title_clean"])
    )

    filtered.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding='utf-8-sig'
    )

    first_chunk = False
    print(f"Rows kept: {len(filtered):,}")

print("Done")


Processing chunk 1
Rows kept: 296,145
Processing chunk 2
Rows kept: 304,475
Processing chunk 3
Rows kept: 293,848
Processing chunk 4
Rows kept: 301,617
Processing chunk 5
Rows kept: 297,911
Processing chunk 6
Rows kept: 300,650
Processing chunk 7
Rows kept: 154,698
Done


In [43]:
cols_to_drop = {
    'song_id'
}

full_data_nolyrics = pd.read_csv(
    "/content/drive/MyDrive/datasets/full_data_nolyrics.csv",
    usecols=lambda c: c not in cols_to_drop
)

In [44]:
full_data_nolyrics.shape

(1949344, 8)

In [45]:
full_data_nolyrics.head()

,user_id,play_count,rating,track_id,artist_name,title,artist_clean,title_clean
0,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRCWHIO128F1488FB7,Daft Punk,Harder Better Faster Stronger,da punk,harder better faster stronger
1,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRFNWEZ128F1452603,Daft Punk,Human After All (Alter Ego Remix ),da punk,human aer all
2,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRWZXDS128F427A71B,The Black Keys,I Got Mine,the black keys,i got mine
3,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRMMKJX128E07818C1,Daft Punk,Face To Face (Cosmo VItelli Remix),da punk,face to face
4,5a905f000fc1ff3df7ca807d57edb608863db05d,2,1.098612,TRKNKEF128F425FD6A,Digitalism,Pogo,digitalism,pogo


### **5.1.2. Lọc điều kiện**

In [46]:
def iterative_filter_cut(
    df,
    min_user=5,
    min_song=5,
    max_user_items=None,
    max_song_users=None
):
    df = df.copy()

    while True:
        n_users_before = df["user_id"].nunique()
        n_songs_before = df["track_id"].nunique()

        user_counts = df.groupby("user_id")["track_id"].nunique()

        valid_users = user_counts[user_counts >= min_user]

        if max_user_items is not None:
            valid_users = valid_users[valid_users <= max_user_items]

        df = df[df["user_id"].isin(valid_users.index)]

        song_counts = df.groupby("track_id")["user_id"].nunique()

        valid_songs = song_counts[song_counts >= min_song]

        if max_song_users is not None:
            valid_songs = valid_songs[valid_songs <= max_song_users]

        df = df[df["track_id"].isin(valid_songs.index)]

        n_users_after = df["user_id"].nunique()
        n_songs_after = df["track_id"].nunique()

        print(
            f"shape={df.shape} | users={n_users_after} | songs={n_songs_after}"
        )

        # Hội tụ
        if n_users_before == n_users_after and n_songs_before == n_songs_after:
            break

    return df


In [47]:
data_cut = iterative_filter_cut(
    full_data_nolyrics,
    min_user=20,
    max_user_items=800,
    min_song=20,
    max_song_users=5000
)

shape=(1887448, 8) | users=19937 | songs=11261
shape=(1886781, 8) | users=19903 | songs=11258
shape=(1886781, 8) | users=19903 | songs=11258


In [48]:
data_cut.to_csv("/content/drive/MyDrive/datasets/data_cut.csv", index=False, encoding='utf-8-sig')

### **5.1.3. Ghép TP và Spotify**

In [49]:
data_cut = pd.read_csv("/content/drive/MyDrive/datasets/data_cut.csv")

In [50]:
data_cut.shape

(1886781, 8)

In [51]:
# artist, name
data_unique = data.drop_duplicates(
    subset=["artist_clean", "name_clean"]
)

# Tạo key tuple
data_unique["key"] = list(
    zip(data_unique["artist_clean"], data_unique["name_clean"])
)

# Map artist_clean, name_clean
data_map = data_unique.set_index("key").to_dict("index")

print("Data map size:", len(data_map))

Data map size: 150540


In [52]:
chunksize = 1_000_000
output_file = "/content/drive/MyDrive/datasets/data_audio.csv"

first_chunk = True
total_rows = len(data_cut)
total_iters = (total_rows + chunksize - 1) // chunksize

for i, start in enumerate(range(0, total_rows, chunksize), start=1):
    end = min(start + chunksize, total_rows)
    chunk = data_cut.iloc[start:end]

    print(f"Processing chunk {i}/{total_iters}")

    # Tạo key join
    keys = list(zip(chunk["artist_clean"], chunk["title_clean"]))

    rows = []

    for row, key in zip(chunk.to_dict("records"), keys):
        data_info = data_map.get(key)
        if data_info is None:
            continue

        merged_row = {}
        merged_row.update(row)        # data (11M)
        merged_row.update(data_info)  # data_map (170K)

        rows.append(merged_row)

    if not rows:
        print(" No rows matched\n")
        continue

    out_df = pd.DataFrame(rows)

    out_df.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8-sig"
    )

    first_chunk = False
    print(f"Rows written: {len(out_df):,}\n")

print("Done")


Processing chunk 1/2
Rows written: 999,555

Processing chunk 2/2
Rows written: 886,377

Done


## **5.2. Ghép User Behavior với Lyrics**

In [53]:
cols_to_drop = {
    'name_clean', 'title_clean'
}

data_audio = pd.read_csv(
    "/content/drive/MyDrive/datasets/data_audio.csv",
    usecols=lambda c: c not in cols_to_drop
)

In [54]:
data_audio.shape

(1885932, 22)

In [55]:
lyrics_map = lyrics.set_index("track_id")["lyrics"].to_dict()

print("Lyrics tracks:", len(lyrics_map))

Lyrics tracks: 237662


In [56]:
chunksize = 1_000_000

input_file = "/content/drive/MyDrive/datasets/data_audio.csv"
output_file = "/content/drive/MyDrive/datasets/data_lyrics.csv"

first_chunk = True
total_rows = len(data_audio)
total_iters = (total_rows + chunksize - 1) // chunksize

for i, chunk in enumerate(
    pd.read_csv(input_file, chunksize=chunksize),
    start=1
):
    print(f"Processing chunk {i}")

    chunk["lyrics"] = chunk["track_id"].map(lyrics_map)

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8-sig"
    )

    first_chunk = False
    print(f"Rows written: {len(chunk):,}\n")

print("Done")


Processing chunk 1
Rows written: 1,000,000

Processing chunk 2
Rows written: 885,932

Done


In [57]:
data_lyrics = pd.read_csv("/content/drive/MyDrive/datasets/data_lyrics.csv")

In [58]:
print("Thống kê sau khi đồng bộ:")
print(f"- Record sau lọc: {len(data_lyrics):,}")
print(f"- User còn lại: {data_lyrics['user_id'].nunique():,}")
print(f"- Bài hát còn lại: {data_lyrics['track_id'].nunique():,}")

Thống kê sau khi đồng bộ:
- Record sau lọc: 1,885,932
- User còn lại: 19,903
- Bài hát còn lại: 11,256


# **6. Cắt giảm user nếu cần**

In [59]:
'''top_users = (
    data_lyrics.groupby('user_id')['song_id']
      .nunique()
      .sort_values(ascending=False)
      .head(20_000)
      .index
)

data_lyrics = data_lyrics[data_lyrics['user_id'].isin(top_users)]'''

"top_users = (\n    data_lyrics.groupby('user_id')['song_id']\n      .nunique()\n      .sort_values(ascending=False)\n      .head(20_000)\n      .index\n)\n\ndata_lyrics = data_lyrics[data_lyrics['user_id'].isin(top_users)]"

In [60]:
'''data_lyrics = iterative_filter_cut(
    data_lyrics,
    min_user=20,
    max_user_items=800,
    min_song=20,
    max_song_users=5000
)'''

'data_lyrics = iterative_filter_cut(\n    data_lyrics,\n    min_user=20,\n    max_user_items=800,\n    min_song=20,\n    max_song_users=5000\n)'

# **7. Data sử dụng**

In [61]:
print("Thống kê sau khi đồng bộ:")
print(f"- Record sau lọc: {len(data_lyrics):,}")
print(f"- User còn lại: {data_lyrics['user_id'].nunique():,}")
print(f"- Bài hát còn lại: {data_lyrics['track_id'].nunique():,}")
size_mb = os.path.getsize("/content/drive/MyDrive/datasets/data_lyrics.csv") / 1024**2
print(f"- File size: {size_mb:.2f} MB")
mem_mb = data_lyrics.memory_usage(deep=True).sum() / 1024**2
print(f"- Memory usage: {mem_mb:.2f} MB")

Thống kê sau khi đồng bộ:
- Record sau lọc: 1,885,932
- User còn lại: 19,903
- Bài hát còn lại: 11,256
- File size: 1763.16 MB
- Memory usage: 2654.37 MB


In [62]:
data_lyrics.head(3)

,user_id,play_count,rating,track_id,artist_name,title,artist_clean,title_clean,valence,year,...,liveness,loudness,popularity,speechiness,tempo,duration_s,main_artist,name_clean,decade,lyrics
0,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRCWHIO128F1488FB7,Daft Punk,Harder Better Faster Stronger,da punk,harder better faster stronger,0.692,2001,...,0.3580,-8.898,71,0.1440,123.475,224.693,Daft Punk,harder better faster stronger,2000s,it it it it it it it it it it it it it it it i...
1,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRFNWEZ128F1452603,Daft Punk,Human After All (Alter Ego Remix ),da punk,human aer all,0.802,2005,...,0.0802,-4.367,46,0.3420,136.251,319.880,Daft Punk,human aer all,2000s,NaN
2,5a905f000fc1ff3df7ca807d57edb608863db05d,1,0.693147,TRWZXDS128F427A71B,The Black Keys,I Got Mine,the black keys,i got mine,0.475,2008,...,0.1730,-6.336,51,0.0406,157.725,239.147,The Black Keys,i got mine,2000s,i i i i i i i i i i i i i i i i i i i i the an...


In [63]:
user_listen_count = (
    data_lyrics.groupby("user_id")["track_id"]
      .nunique()
      .reset_index(name="num_songs")
)

user_most = user_listen_count.sort_values(
    "num_songs", ascending=False
).head(10)

print("User nghe nhiều bài nhất")
print(user_most)

user_least = user_listen_count.sort_values(
    "num_songs", ascending=True
).head(10)

print("User nghe ít bài nhất")
print(user_least)

User nghe nhiều bài nhất
                                        user_id  num_songs
6085   4e73d9e058d2b1f2dba9c1fe4a8f416f9f58364f        732
10904  8cb51abc6bf8ea29341cb070fe1e1af5e4c3ffcc        583
926    0c2932cb475b83b61039bdfbb72c14580b8fad2b        525
1351   119b7c88d58d0c6eb051365c103da5caf817bea6        523
15036  c1255748c06ee3f6440c51c439446886c7807095        514
15864  cbc7bddbe3b2f59fdbe031b3c8d0db4175d361e6        504
14295  b7c24f770be6b802805ac0e2106624a517643c17        464
17043  db6a78c78c9239aba33861dae7611a6893fb27d5        448
8402   6d625c6557df84b60d90426c0116138b617b9449        439
18622  ef6f8b6e34b0305ba4c77e7d56390cd2e94c60d4        437
User nghe ít bài nhất
                                        user_id  num_songs
9276   7800d3894ccd3e687aaa3d9824707aca1a76aac4         19
5758   4a3c29a00b09fc26fccf843c8c52bdd74f19cf32         20
3736   2ffe0bbf01ee2127507d8f041d5776cc12f845ea         20
691    095f4a4ab9d82f20efdc731e064d872ce7bc86d8         20
17388  e0

In [64]:
song_listen_count = (
    data_lyrics.groupby("track_id")["user_id"]
      .nunique()
      .reset_index(name="num_listens")
)

song_most = song_listen_count.sort_values(
    "num_listens", ascending=False
).head(10)

print("Bài được nghe nhiều nhất")
print(song_most)

song_least = song_listen_count.sort_values(
    "num_listens", ascending=True
).head(10)

print("Bài được nghe ít nhất")
print(song_least)

Bài được nghe nhiều nhất
                track_id  num_listens
3157  TRHKJNX12903CEFCDF         4424
3515  TRIEXMF128F92FDD60         3650
6246  TRONYHY128F92C9D11         3545
1922  TRENTGL128E0780C8E         3447
6323  TROTIUH128E0782538         3296
6984  TRQFXKD128E0780CAE         3262
3598  TRIKGRK128E0780DB0         3260
6031  TROAQBZ128F9326213         3108
5920  TRNTALF128EF343800         2930
2938  TRGXQES128F42BA5EB         2874
Bài được nghe ít nhất
                 track_id  num_listens
6745   TRPRBIM128F932E112           20
6793   TRPTWQB128F4243801           20
1811   TREGJYU128E078FD2F           20
8498   TRTTQTD128F422B032           20
9244   TRVLIMG128E078D2AB           20
10688  TRYSXMJ128F935C9C6           20
7933   TRSJRRS128F429808F           20
3112   TRHHNQB128F4248478           20
11151  TRZUCPT128F42654FB           20
7434   TRRGIWA128F426610C           20
